In [ ]:
from typing import cast

import torch
import numpy as np
import plotly.express as px
from darts.models import SKLearnModel
from darts.explainability.shap_explainer import ShapExplainer
from sklearn.linear_model import LinearRegression

from aare_train.fetching.feature_set import FeatureSet
from aare_train.features.registry import FEATURES
from aare_train.params import read_params
from aare_train.storage.model import load_model

# Explainability analysis of LR model

Using the shap integration in darts and the learned coefficients we can see which features are most important for the resulting forecast.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()

In [ ]:
meta, model, scalers = load_model(name="LR", version="dev")
assert isinstance(model, SKLearnModel)

In [ ]:
ds = FeatureSet(
    targets=FEATURES.get_many(meta["features"]["targets"]),
    future=FEATURES.get_many(meta["features"].get("future")),
    split_params=params["split"],
)

In [ ]:
train = ds.get_train()
train_target_subs = train[0]
train_fc_subs = train[2]

In [ ]:
assert scalers is not None
scaler_target = scalers.get("series")
scaler_fc = scalers.get("future_covariates")

assert scaler_target is not None
assert scaler_fc is not None

explainer = ShapExplainer(
    model,
    background_series=scaler_target.transform(train_target_subs),
    background_future_covariates=scaler_fc.transform(train_fc_subs),
)

explainer.summary_plot()

In [ ]:
sk_model = cast(LinearRegression, model.model)
sk_model

In [ ]:
sort_idx = np.argsort(np.abs(sk_model.coef_))
names = np.array(model.lagged_feature_names)[sort_idx]
coefs = sk_model.coef_[sort_idx]

px.bar(y=names, x=coefs)

I looked at two random models both with <0.31 MAE and one was heavily focused on relative humidty while the other one was focused almost exclusively on air temperature (makes sense, a model only using air temp also reaches MAE 0.31). From now on it's only about the latter model: \
The only other features in the top 20 are rainfall (rr) and wind_x.
Looking at the most influential features we see that lag 0 and 1 are by far the most imporant ones. Lag 4 also has some applications, but higher lags only provide minimal corrections.
Regarding transformations, it seems like ma3 and ma6 are quite popular for both tt and rr. Other than that, only diff makes an appearance.

Might be interesting to see if a model with just

- tt, rr (or rh)
- raw, ma3, ma6
- lag 0, 1, 4

could achieve comparable performance. If yes, it might be a good fit for a probabilistic model because there it's much nicer to have few features. The idea is to generalize better and get confidence intervals at the cost of higher val errors.